# Phase 3: Functional Enrichment

This notebook separates the enrichment analysis from the earlier combined notebook and starts from the Phase 2 DEG outputs saved under `results/`.

Current scope in this repo:
- GO Biological Process enrichment for each breast cancer subtype
- subtype-specific dot plots for activated and suppressed GO terms

Deferred for now:
- KEGG enrichment
- KEGG pathway map overlays


## Inputs And Outputs

Expected inputs:
- `results/DEGs_Basal_vs_Normal.csv`
- `results/DEGs_Her2_vs_Normal.csv`
- `results/DEGs_LumA_vs_Normal.csv`
- `results/DEGs_LumB_vs_Normal.csv`

Outputs written by this notebook:
- `results/phase3/go_bp_Basal_activated.csv`
- `results/phase3/go_bp_Basal_suppressed.csv`
- `results/phase3/go_bp_Her2_activated.csv`
- `results/phase3/go_bp_Her2_suppressed.csv`
- `results/phase3/go_bp_LumA_activated.csv`
- `results/phase3/go_bp_LumA_suppressed.csv`
- `results/phase3/go_bp_LumB_activated.csv`
- `results/phase3/go_bp_LumB_suppressed.csv`
- `results/phase3/go_bp_all_subtypes.csv`
- `figures/phase3_go_bp_Basal_dot.png`
- `figures/phase3_go_bp_Her2_dot.png`
- `figures/phase3_go_bp_LumA_dot.png`
- `figures/phase3_go_bp_LumB_dot.png`

Note: running Enrichr through `gseapy` requires internet access.


In [ ]:
from pathlib import Path
import os
import warnings

mpl_cache = Path.cwd() / ".mplconfig"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

try:
    import gseapy as gp
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gseapy"])
    import gseapy as gp

FIG_DIR = Path("figures")
RES_DIR = Path("results/phase3")
FIG_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

TUMOR_SUBTYPES = ["Basal", "Her2", "LumA", "LumB"]
GO_LIBRARY = "GO_Biological_Process_2023"

required_deg_files = {
    subtype: Path(f"results/DEGs_{subtype}_vs_Normal.csv")
    for subtype in TUMOR_SUBTYPES
}
missing = [str(path) for path in required_deg_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing Phase 2 DEG outputs: " + ", ".join(missing) +
        ". Run phase2_differential_expression.ipynb first."
    )


## Load Significant DEG Tables

We use the significant subtype-vs-normal DEG lists from Phase 2 as the input gene sets for enrichment.


In [ ]:
sig_deg = {
    subtype: pd.read_csv(path)
    for subtype, path in required_deg_files.items()
}

deg_gene_sets = {
    subtype: sorted(sig_deg[subtype]["gene_name"].dropna().astype(str).unique().tolist())
    for subtype in TUMOR_SUBTYPES
}

pd.DataFrame(
    {
        "Subtype": TUMOR_SUBTYPES,
        "Genes in DEG set": [len(deg_gene_sets[subtype]) for subtype in TUMOR_SUBTYPES],
    }
).set_index("Subtype")


## GO Biological Process Enrichment

Each subtype is split into activated and suppressed genes using the sign of the log2 fold change, then enriched against `GO_Biological_Process_2023` with Enrichr via `gseapy`.


In [ ]:
go_results = []

for subtype in TUMOR_SUBTYPES:
    df_sig = sig_deg[subtype].copy()

    up_genes = df_sig.loc[df_sig["log2FoldChange"] > 0, "gene_name"].dropna().astype(str).unique().tolist()
    down_genes = df_sig.loc[df_sig["log2FoldChange"] < 0, "gene_name"].dropna().astype(str).unique().tolist()

    for direction, genes in [("Activated", up_genes), ("Suppressed", down_genes)]:
        if len(genes) < 10:
            print(f"Skipping {subtype} {direction}: too few genes for GO BP ({len(genes)}).")
            continue

        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=[GO_LIBRARY],
            organism="human",
            outdir=None,
            cutoff=0.5,
        )

        df = enr.results.copy()
        df["Subtype"] = subtype
        df["Direction"] = direction

        if "Overlap" not in df.columns and "Genes" in df.columns:
            df["Overlap"] = df["Genes"].astype(str).apply(
                lambda value: str(len(value.split(";")) if value else 0) + "/0"
            )

        if df["Adjusted P-value"].isna().all() and "Old adjusted P-value" in df.columns:
            df["Adjusted P-value"] = df["Old adjusted P-value"]
            if "Old P-value" in df.columns:
                df["P-value"] = df["Old P-value"]

        df = df.sort_values("Adjusted P-value", ascending=True)
        df.to_csv(RES_DIR / f"go_bp_{subtype}_{direction.lower()}.csv", index=False)
        go_results.append(df)

go_all = pd.concat(go_results, ignore_index=True) if go_results else pd.DataFrame()
if not go_all.empty:
    go_all.to_csv(RES_DIR / "go_bp_all_subtypes.csv", index=False)
    print(f"Saved GO BP enrichment: {go_all.shape[0]:,} rows total")
else:
    print("No GO BP enrichment results generated.")


## Plot The Enriched GO Terms

For each subtype, the notebook creates a paired dot plot showing the strongest activated and suppressed GO Biological Process terms.


In [ ]:
if go_all.empty:
    print("No GO BP results to plot.")
else:
    for subtype in TUMOR_SUBTYPES:
        sub_df = go_all[go_all["Subtype"] == subtype].copy()
        if sub_df.empty:
            print(f"Skipping {subtype}: no GO BP terms available.")
            continue

        sub_df["Count"] = sub_df["Overlap"].astype(str).str.split("/").str[0].astype(float)
        sub_df["neg_log10_padj"] = -np.log10(sub_df["Adjusted P-value"].clip(lower=1e-300))

        activated = (
            sub_df[sub_df["Direction"] == "Activated"]
            .sort_values("Adjusted P-value", ascending=True)
            .head(10)
            .copy()
        )
        suppressed = (
            sub_df[sub_df["Direction"] == "Suppressed"]
            .sort_values("Adjusted P-value", ascending=True)
            .head(10)
            .copy()
        )

        if activated.empty and suppressed.empty:
            print(f"Skipping {subtype}: no activated/suppressed GO terms available.")
            continue

        fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=False)
        panels = [(axes[0], activated, "Activated"), (axes[1], suppressed, "Suppressed")]
        scatter = None

        for ax, panel_df, label in panels:
            if panel_df.empty:
                ax.set_title(label)
                ax.text(0.5, 0.5, "No terms", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue

            panel_df = panel_df.sort_values("neg_log10_padj", ascending=True)
            y = np.arange(len(panel_df))
            scatter = ax.scatter(
                panel_df["neg_log10_padj"],
                y,
                s=panel_df["Count"] * 8,
                c=panel_df["neg_log10_padj"],
                cmap="plasma",
                alpha=0.9,
                edgecolor="black",
                linewidth=0.4,
            )
            ax.set_yticks(y)
            ax.set_yticklabels(panel_df["Term"], fontsize=8)
            ax.set_xlabel("-log10(Adjusted P-value)")
            ax.set_title(label)
            ax.grid(alpha=0.25)

        fig.suptitle(f"{subtype}: GO Biological Process Enrichment", y=1.02, fontsize=14)
        fig.tight_layout()

        if scatter is not None:
            cbar = fig.colorbar(scatter, ax=axes, shrink=0.8, pad=0.02)
            cbar.set_label("Significance: -log10(adj p)")

        out_path = FIG_DIR / f"phase3_go_bp_{subtype}_dot.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved GO BP dot plot: {out_path}")


## Interpretation Prompts

When you review the Phase 3 outputs, focus on:
- which biological programs are activated versus suppressed in each subtype
- whether Basal, Her2, LumA, and LumB show clearly different GO themes
- which terms can help explain the hub-gene communities from Phase 4
- which top enriched processes should be highlighted in the final report

A practical next writing step is to take the top 3 to 5 activated and suppressed GO terms per subtype from `results/phase3/go_bp_all_subtypes.csv` and turn them into a short biological summary.
